# 01 — Build model-only bundle (LOCAL, không retrieval)

Notebook này tạo đúng 100 prompt gồm **câu hỏi + dữ liệu lá số gốc** để chạy baseline chỉ dùng mô hình. Không cần Neo4j, không chạy graph/dense/sparse retrieval, fusion, reranker, grading hay trích xuất chart facts. ZIP đầu ra được upload thành private Kaggle Dataset cho notebook 02.


In [ ]:
from pathlib import Path
import sys

# Nếu auto-detect không được, điền đường dẫn repo tuyệt đối vào đây.
REPO_ROOT = None
RUN_MODE = 'smoke'  # smoke | official

def find_repo(explicit=None):
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        assert (candidate / 'backend' / 'app').exists(), candidate
        return candidate
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    direct = [p for p in starts if (p / 'backend' / 'app').exists()]
    children = [p / 'tuvi-battu-graphrag' for p in starts if (p / 'tuvi-battu-graphrag' / 'backend' / 'app').exists()]
    matches = sorted(set(direct + children))
    assert len(matches) == 1, f'Hãy đặt REPO_ROOT; tìm thấy {matches}'
    return matches[0]

REPO_ROOT = find_repo(REPO_ROOT)
KIT_ROOT = REPO_ROOT / 'benchmark' / 'tuvi_golden_dataset' / 'local_llm_ablation'
sys.path.insert(0, str(KIT_ROOT))
assert RUN_MODE in {'smoke', 'official'}
print({'repo': str(REPO_ROOT), 'kit': str(KIT_ROOT), 'mode': RUN_MODE})


In [ ]:
is_official = RUN_MODE == 'official'
output_name = 'model_only_bundle_v1' if is_official else 'model_only_bundle_v1_smoke'
BUNDLE_CONFIG = {
    'repo_root': str(REPO_ROOT),
    'dataset_path': str(REPO_ROOT / 'benchmark' / 'tuvi_golden_dataset' / 'release' / 'tuviqa_v1_release.jsonl'),
    'item_limit': None if is_official else 2,
    'output_dir': str(KIT_ROOT / 'artifacts' / output_name),
}
from local_tools.build_model_only_bundle import build_model_only_bundle
manifest = build_model_only_bundle(BUNDLE_CONFIG)
manifest


In [ ]:
expected = 100 if is_official else 2
assert manifest['bundle_type'] == 'model_only_question_plus_raw_chart', manifest
assert manifest['selected_suites'] == ['model_only'], manifest
assert manifest['config_count'] == 1, manifest
assert manifest['planned_pair_count'] == expected, manifest
assert manifest['completed_pair_count'] == expected, manifest
assert manifest['failed_pair_count'] == 0, manifest
assert manifest['is_complete'], manifest
assert manifest['retrieval_executed'] is False, manifest
assert manifest['derived_chart_features_used'] is False, manifest
assert manifest['corpus_context_used'] is False, manifest

import shutil
bundle_dir = Path(BUNDLE_CONFIG['output_dir'])
archive = shutil.make_archive(str(bundle_dir), 'zip', root_dir=bundle_dir)
print('PASS — upload ZIP này thành private Kaggle Dataset:', archive)
